In [34]:
import os
from dotenv import load_dotenv
import xarray as xr

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datetime import date

import analysis_utils
import isku_utils

import importlib

importlib.reload(analysis_utils)
importlib.reload(isku_utils)

<module 'isku_utils' from '/home/emily_zuetell/projects/poreallas/analysis/isku_utils.py'>

In [42]:
load_dotenv()
DATA_DIR = os.environ["DATA_DIR"]
# Baseline period for Impact
BASELINE_PERIOD = slice("1996-01-01", "2025-12-31")
# Define Forecast Months
FC_MONTHS = [9, 10, 11, 12, 1, 2]
FC_PERIOD = slice("2026-09-01", "2027-02-28")

config = analysis_utils.ImpactConfig( version = "v260910",
                                     baseline_period=BASELINE_PERIOD, 
                                     rate=False, 
                                     months = FC_MONTHS,
                                     hotonly = "hotonly", 
                                     dims = ['number', 'sample'])

# Define Forecast
EFFECTS_URI = "/home/emily_zuetell/projects/poreallas/data/v20260909_effects_with_betas.zarr"

In [43]:
# Projection Effects
effect = xr.open_datatree(os.path.join(DATA_DIR, EFFECTS_URI), consolidated=False)

In [44]:
### Log baseline period and impact calculation
rate_l = "rate" if config.rate else "total"
baseline_tag = analysis_utils._baseline_tag(config.baseline_period)

In [45]:
# Compute impact: forecast - baseline
impact = config.compute_impact(effect.chunk({dim: -1 for dim in config.dims}), ensemble=True) #Maintain individual ensemble members
# Use only the defined 6-months
impact = impact.sel(month=config.months)

In [46]:
# Aggregate Impact Regions to group_level
group_level = 'ISO' #IR: Impact region, # ADM1: State level, # ISO: Country level
impact, merge_key, base_cols = analysis_utils.aggregate_impact(impact, config, group_level)

In [ ]:
#Compute stats in xarray from dims in config.dims
stat_cols = ["median", "p17", "p83", "likely_range_IPCC", "mean", "std", "min", "max", "p10", "p90"]
_polygons_impact = analysis_utils.dataset_to_dataframe(analysis_utils.compute_stats(impact, dim=config.dims))

In [ ]:
# Format dataframe for csv output       
wide = _polygons_impact.pivot(
    index=base_cols,
    columns="month", values=stat_cols,
)
wide.columns = [f"month {m} {stat}" for stat, m in wide.columns]
stat_col_names = wide.columns.difference(base_cols)
wide[stat_col_names] = wide[stat_col_names].round(0).astype("Int64")
wide = wide.reset_index()

In [ ]:
# Output CSV

# Log parameters in filename
filename_template="{version}_{hotonly}_{scope}_{rate_l}_{stat_scope}_{group_level}_{baseline}.csv"

wide.to_csv(
    filename_template.format(
        version=config.version,
        hotonly=config.hotonly,
        rate_l=rate_l,
        scope="monthly",
        stat_scope="",
        group_level=group_level,
        baseline=baseline_tag,
    ),
    index=False,
)